# 07 — Step 6: Circuit Composition Analysis

**Goal**: Investigate whether confirmed reflection heads work together as a circuit
by comparing joint ablation effects vs the sum of individual effects.

If joint ablation causes *more* degradation than the sum of individual ablations,
the heads are **superadditive** — they form a synergistic circuit.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/iker0/mi-mirror/blob/main/notebook_demo/07_step6_circuit_composition.ipynb
)

In [ ]:
# Clone repo and set up paths (run once on Colab)
!git clone https://github.com/iker0/mi-mirror.git /content/mi-mirror 2>/dev/null || echo "Already cloned"
%cd /content/mi-mirror

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from itertools import combinations
from collections import OrderedDict

from scripts.config import (
    EXPERIMENTS_DIR, MIRROR_PROMPTS, SEEDS, MIRROR_DIR, PROMPT_OBJECTS,
    NUM_DUAL_STREAM_BLOCKS,
)
from scripts.model_loader import load_flux_pipeline
from scripts.roi import get_object_and_reflection_rois
from scripts.generate import generate_with_ablation, generate_baseline
from scripts.metrics import compute_reflection_quality, compute_superadditivity
from scripts.visualization import (
    plot_ablation_comparison,
    plot_superadditivity_heatmap,
    plot_circuit_diagram,
)

STEP5_DIR = EXPERIMENTS_DIR / "step5_causal"
STEP6_DIR = EXPERIMENTS_DIR / "step6_circuit"
STEP6_DIR.mkdir(parents=True, exist_ok=True)

# Load step 5 results
with open(STEP5_DIR / "step5_results.pkl", "rb") as f:
    step5 = pickle.load(f)

confirmed_heads = step5["confirmed_heads"]
avg_metrics_s5 = step5["avg_metrics"]

print(f"Confirmed reflection heads: {confirmed_heads}")
print(f"Individual ablation metrics available for: {list(avg_metrics_s5.keys())}")

In [ ]:
pipe = load_flux_pipeline(quantize_nf4=True, cpu_offload=True)
print("Model loaded.")

In [ ]:
# Use same test configs as step 5
test_configs = step5["test_configs"]

# Generate baselines and get ROIs
baselines = {}
rois = {}
for prompt_idx, seed in test_configs:
    prompt = MIRROR_PROMPTS[prompt_idx]
    obj_name = PROMPT_OBJECTS[prompt_idx]
    
    img_path = MIRROR_DIR / f"prompt{prompt_idx:02d}_seed{seed}.png"
    if img_path.exists():
        ref_image = Image.open(img_path)
    else:
        ref_image = generate_baseline(pipe, prompt, seed)
    
    obj_roi, ref_roi = get_object_and_reflection_rois(ref_image, obj_name)
    baseline = generate_baseline(pipe, prompt, seed)
    baselines[(prompt_idx, seed)] = baseline
    rois[(prompt_idx, seed)] = (obj_roi, ref_roi)

print(f"Prepared {len(baselines)} baselines.")

In [ ]:
# Individual ablation MSE values (re-use from step 5 or recompute)
# We need the MSE_reflection per head as the "individual effect"

individual_effects = {}  # (b, h) -> mean MSE_reflection
for (b, h, _) in confirmed_heads:
    key = f"B{b}H{h}"
    if key in avg_metrics_s5:
        individual_effects[(b, h)] = avg_metrics_s5[key]["mse_reflection"]
    else:
        # Recompute if not in step5 results
        effects = []
        for prompt_idx, seed in test_configs:
            prompt = MIRROR_PROMPTS[prompt_idx]
            ablated = generate_with_ablation(pipe, prompt, seed, [(b, [h])])
            obj_roi, ref_roi = rois[(prompt_idx, seed)]
            m = compute_reflection_quality(baselines[(prompt_idx, seed)], ablated, ref_roi, obj_roi)
            effects.append(m["mse_reflection"])
        individual_effects[(b, h)] = float(np.mean(effects))

print("Individual effects (MSE_reflection):")
for (b, h), eff in individual_effects.items():
    print(f"  B{b}H{h}: {eff:.6f}")

In [ ]:
# Pairwise ablation: ablate each pair of confirmed heads simultaneously
# Group by layer depth: early (dual-stream, <19) vs late (single-stream, >=19)

head_list = [(b, h) for b, h, _ in confirmed_heads]
pairs = list(combinations(head_list, 2))

pairwise_joint_effects = {}  # ((b1,h1), (b2,h2)) -> mean MSE_reflection
pairwise_images = {}  # for visualization

print(f"Testing {len(pairs)} pairwise ablations...")

for (b1, h1), (b2, h2) in pairs:
    print(f"\n  Ablating B{b1}H{h1} + B{b2}H{h2}")
    joint_effects = []
    
    # Build ablation targets: group heads by block
    if b1 == b2:
        ablation_targets = [(b1, [h1, h2])]
    else:
        ablation_targets = [(b1, [h1]), (b2, [h2])]
    
    for prompt_idx, seed in test_configs:
        prompt = MIRROR_PROMPTS[prompt_idx]
        ablated = generate_with_ablation(pipe, prompt, seed, ablation_targets)
        obj_roi, ref_roi = rois[(prompt_idx, seed)]
        m = compute_reflection_quality(baselines[(prompt_idx, seed)], ablated, ref_roi, obj_roi)
        joint_effects.append(m["mse_reflection"])
        
        # Save first config's image for visualization
        if prompt_idx == test_configs[0][0] and seed == test_configs[0][1]:
            pairwise_images[((b1, h1), (b2, h2))] = ablated
    
    mean_joint = float(np.mean(joint_effects))
    pairwise_joint_effects[((b1, h1), (b2, h2))] = mean_joint
    print(f"    Joint MSE_ref: {mean_joint:.6f}")

print("\nPairwise ablation complete.")

In [ ]:
# Superadditivity analysis
pairwise_superadditivity = {}

print("=== Superadditivity Analysis ===")
print(f"{'Pair':>20} {'Individual Sum':>15} {'Joint Effect':>15} {'Superadditivity':>15}")

for ((b1, h1), (b2, h2)), joint in pairwise_joint_effects.items():
    ind1 = individual_effects[(b1, h1)]
    ind2 = individual_effects[(b2, h2)]
    sa = compute_superadditivity([ind1, ind2], joint)
    pairwise_superadditivity[((b1, h1), (b2, h2))] = sa
    
    label = f"B{b1}H{h1}+B{b2}H{h2}"
    print(f"{label:>20} {ind1+ind2:>15.6f} {joint:>15.6f} {sa:>15.6f}")

# Interpretation
synergistic = {k: v for k, v in pairwise_superadditivity.items() if v > 0}
print(f"\n{len(synergistic)}/{len(pairwise_superadditivity)} pairs are synergistic (superadditive > 0)")

In [ ]:
# Visualize: superadditivity heatmap
if pairwise_superadditivity:
    fig = plot_superadditivity_heatmap(
        pairwise_superadditivity,
        title="Pairwise Superadditivity (MSE_reflection)",
    )
    fig.savefig(STEP6_DIR / "superadditivity_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Circuit diagram
if pairwise_superadditivity:
    fig = plot_circuit_diagram(
        pairwise_superadditivity,
        threshold=0.0,
        title="Reflection Head Circuit (edges = synergistic pairs)",
    )
    fig.savefig(STEP6_DIR / "circuit_diagram.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Side-by-side: individual vs joint ablation images
prompt_idx, seed = test_configs[0]
original = baselines[(prompt_idx, seed)]

if pairwise_images:
    # Show the most synergistic pair
    best_pair = max(pairwise_superadditivity, key=pairwise_superadditivity.get)
    (b1, h1), (b2, h2) = best_pair
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(original)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    # Individual ablations (re-generate for display)
    prompt = MIRROR_PROMPTS[prompt_idx]
    img1 = generate_with_ablation(pipe, prompt, seed, [(b1, [h1])])
    axes[1].imshow(img1)
    axes[1].set_title(f"Ablate B{b1}H{h1}")
    axes[1].axis("off")
    
    img2 = generate_with_ablation(pipe, prompt, seed, [(b2, [h2])])
    axes[2].imshow(img2)
    axes[2].set_title(f"Ablate B{b2}H{h2}")
    axes[2].axis("off")
    
    axes[3].imshow(pairwise_images[best_pair])
    axes[3].set_title(f"Ablate Both (SA={pairwise_superadditivity[best_pair]:.4f})")
    axes[3].axis("off")
    
    fig.suptitle(f"Most Synergistic Pair: B{b1}H{h1} + B{b2}H{h2}", fontsize=14)
    fig.tight_layout()
    fig.savefig(STEP6_DIR / "joint_vs_individual.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Circuit narrative
print("=== Circuit Composition Summary ===")
print()

early_heads = [(b, h) for b, h, _ in confirmed_heads if b < NUM_DUAL_STREAM_BLOCKS]
late_heads = [(b, h) for b, h, _ in confirmed_heads if b >= NUM_DUAL_STREAM_BLOCKS]

print(f"Early-layer heads (dual-stream, blocks 0-{NUM_DUAL_STREAM_BLOCKS-1}): "
      f"{[f'B{b}H{h}' for b, h in early_heads]}")
print(f"Late-layer heads (single-stream, blocks {NUM_DUAL_STREAM_BLOCKS}-56): "
      f"{[f'B{b}H{h}' for b, h in late_heads]}")

# Check early-late cooperation
early_late_pairs = {
    k: v for k, v in pairwise_superadditivity.items()
    if (k[0] in early_heads and k[1] in late_heads) or
       (k[0] in late_heads and k[1] in early_heads)
}
if early_late_pairs:
    avg_el_sa = np.mean(list(early_late_pairs.values()))
    print(f"\nEarly-Late cross-layer superadditivity: {avg_el_sa:.6f} (mean of {len(early_late_pairs)} pairs)")
    if avg_el_sa > 0:
        print("  -> Evidence of cross-layer circuit cooperation!")
    else:
        print("  -> Heads appear to work independently across layers.")

print("\n=== Step 6 Complete ===")

In [ ]:
# Save results
results = {
    "confirmed_heads": confirmed_heads,
    "individual_effects": {f"B{b}H{h}": v for (b, h), v in individual_effects.items()},
    "pairwise_joint_effects": {
        f"B{b1}H{h1}+B{b2}H{h2}": v
        for ((b1, h1), (b2, h2)), v in pairwise_joint_effects.items()
    },
    "pairwise_superadditivity": {
        f"B{b1}H{h1}+B{b2}H{h2}": v
        for ((b1, h1), (b2, h2)), v in pairwise_superadditivity.items()
    },
    "test_configs": test_configs,
}
with open(STEP6_DIR / "step6_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Results saved to {STEP6_DIR}")